# Activity 004: DNA Detective
In this final activity, we will put together everything we learned in the previous lessons and activities. We will use the 3 techniques we've learned to identify a piece of mystery DNA by comparing it to a database of known organims' DNA.

Remember, the main techniques that we covered are:
1. Identifying and comparing genomic signaturesusing: GC content and k-mers.

2. Comparing the sequences directly using Global Sequence Alignment with a prescribed scoring scheme.

## 1. Install the Python library to read the FASTA files

In [1]:
%pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 19.2 MB/s eta 0:00:00


## 2. Grab the functions from previous activities

In [99]:
from Bio import SeqIO
from google.colab import drive
import numpy as np
from collections import defaultdict
drive.mount('/content/drive')

#============================
# FASTA file Functions
def get_fasta_info(file_path):
    # Open the FASTA file
    with open(file_path, "r", encoding="utf-8") as fasta_file:
        record = SeqIO.read(fasta_file, "fasta")  # read in the FASTA records
        print(f"Description: {record.description}")
        print(f"Sequence: {record.seq[:20]}...")  # Print first 20 bases
        print(f"Length: {len(record)}\n")

def get_dna_seq(file_path):
    # Open the FASTA file
    with open(file_path, "r", encoding="utf-8") as fasta_file:
        record = SeqIO.read(fasta_file, "fasta")
        return record.seq

#============================
# GC content Functions
def get_nucleotide_count(dna_seq, nucleotide):
    return dna_seq.count(nucleotide)

def get_GC_content(dna_seq):
    g_count = get_nucleotide_count(dna_seq, "G")
    c_count = get_nucleotide_count(dna_seq, "C")
    dna_length = len(dna_seq)
    return ((g_count + c_count) / dna_length) * 100

def print_gc_content(dna_seq, organism):
    blue_font = "\033[34m"
    reset_font = "\033[0m"
    gc_content = get_GC_content(dna_seq)
    print(f"{organism}: {blue_font}{gc_content:.2f}%{reset_font}")

#============================
# k-mer Functions
def print_kmer_count(kmer_count, organism, extra_info: str = ""):
    blue_font = "\033[34m"
    reset_font = "\033[0m"
    if extra_info:
        extra_info_str = f" ({extra_info})"
    else:
        extra_info_str = ""
    print(f"{organism}: {blue_font}{kmer_count:.2f}%{reset_font} {extra_info_str}")

def count_kmers(dna_seq, k):
    """Returns the kmer counts dictionary and the total
       number of kmers in the sequence."""
    kmer_counts = defaultdict(int)
    for i in range(len(dna_seq) - k):
        kmer = dna_seq[i:i+k]
        kmer_counts[kmer] += 1

    return kmer_counts, len(dna_seq) - k

def get_kmer_percentage(kmer_count, total_kmer_count):
  return (kmer_count / total_kmer_count) * 100

#============================
# Hamming Distance
def get_hamming_distance(seq_1, seq_2):
    hamming_distance = 0
    seq_1_print = []
    seq_2_print = []
    red_font = "\033[31m"
    reset_font = "\033[0m"

    for char_1, char_2 in zip(seq_1, seq_2):
        if char_1 != char_2:
            hamming_distance += 1

            # Rewrite both sequences with red font where the characters differ
            seq_1_print.append(f"{red_font}{char_1}{reset_font}")
            seq_2_print.append(f"{red_font}{char_2}{reset_font}")

        else:
            seq_1_print.append(char_1)
            seq_2_print.append(char_2)

    seq_1_print = "".join(seq_1_print)
    seq_2_print = "".join(seq_2_print)

    return hamming_distance, seq_1_print, seq_2_print

#============================
# Global Sequence Alignment
GAP = "-"

def f(char_1: str, char_2: str) -> int:
    if char_1 == GAP or char_2 == GAP:
        return GAP_PENALTY
    elif char_1 == char_2:
        return MATCH
    else:
        return MISMATCH

def get_dp_matrices(S: str, T: str) -> tuple[np.ndarray, np.ndarray]:
    m = len(S)
    n = len(T)
    scores = np.zeros((m + 1, n + 1), dtype=int)
    pointers = np.zeros((m + 1, n + 1), dtype=int)

    # set top-left corner of pointers matrix to
    # having no further pointers
    pointers[0, 0] = 3

    # setting negative decrements for each '-' that
    # is matched with a character in S or T
    for i in range(1, m+1):
        scores[i,0] = scores[i-1, 0] + f(S[i-1], GAP)
        pointers[i,0] = 1

    for i in range(1, n+1):
        scores[0, i] = scores[0, i-1] + f(GAP, T[i-1])
        pointers[0, i] = 2

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            # calculate all possible scores
            score_options = np.array([
                scores[i - 1, j - 1] + f(S[i - 1], T[j - 1]), # diagonal
                scores[i - 1, j] + f(S[i - 1], GAP),          # up
                scores[i, j - 1] + f(GAP, T[j - 1]),          # left
            ])

            # find the largest score
            scores[i, j] = np.max(score_options)

            # find and set the parent of largest score
            pointers[i, j] = int(np.argmax(score_options))

    return scores, pointers

def backtrack(
    i: int,
    j: int,
    pointers: np.ndarray,
    S: str,
    T: str,
) -> tuple[str, str]:
    # Initialize the S and T alignments as lists of strings
    # The characters from S and T will be added to these lists
    # from right to left
    S_aligned = []
    T_aligned = []

    # Recursive Case
    # Make your way from the bottom-right corner of the pointers matrix
    # up to the top-left corner
    while pointers[i, j] != 3:
        if pointers[i, j] == 0:
            S_aligned.append(S[i - 1])
            T_aligned.append(T[j - 1])
            i -= 1
            j -= 1
        elif pointers[i, j] == 1:
            S_aligned.append(S[i - 1])
            T_aligned.append(GAP)
            i -= 1
        elif pointers[i, j] == 2:
            S_aligned.append(GAP)
            T_aligned.append(T[j - 1])
            j -= 1

    # Reverse the strings
    S_aligned = "".join(reversed(S_aligned))
    T_aligned = "".join(reversed(T_aligned))

    return S_aligned, T_aligned

def get_alignment(S: str, T: str) -> tuple[str, str, int]:
    m = len(S)
    n = len(T)

    # run the alignment score calculations
    scores, pointers = get_dp_matrices(S, T)

    # max score is at the bottom-right corner of the scores matrix
    max_score = int(scores[m, n])

    # call the backtrack function starting at the bottom-right corner
    S_aligned, T_aligned = backtrack(m, n, pointers, S, T)

    return S_aligned, T_aligned, max_score

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 3. Create the DNA database

In [ ]:
base_path = "/content/drive/MyDrive/..."   # REPLACE with your path

actino_fasta_path = f"{base_path}/data/actinobacteria_dna.fasta"
cat_fasta_path = f"{base_path}/data/cat_dna.fasta"
dog_fasta_path = f"{base_path}/data/dog_dna.fasta"
ecoli_fasta_path = f"{base_path}/data/e_coli_dna.fasta"
fruit_fly_fasta_path = f"{base_path}/data/fruit_fly_dna.fasta"
horse_fasta_path = f"{base_path}/data/horse_dna.fasta"
human_fasta_path = f"{base_path}/data/human_dna.fasta"
mouse_fasta_path = f"{base_path}/data/mouse_dna.fasta"
plasmodium_fasta_path = f"{base_path}/data/plasmodium_falciparum_dna.fasta"
thale_cress_fasta_path = f"{base_path}/data/thale_cress_dna.fasta"

# Store the DNA sequences
actino_dna = get_dna_seq(actino_fasta_path)
cat_dna = get_dna_seq(cat_fasta_path)
dog_dna = get_dna_seq(dog_fasta_path)
ecoli_dna = get_dna_seq(ecoli_fasta_path)
fruit_fly_dna = get_dna_seq(fruit_fly_fasta_path)
horse_dna = get_dna_seq(horse_fasta_path)
human_dna = get_dna_seq(human_fasta_path)
mouse_dna = get_dna_seq(mouse_fasta_path)
plasmodium_dna = get_dna_seq(plasmodium_fasta_path)
thale_cress_dna = get_dna_seq(thale_cress_fasta_path)

# Set the DNA database
dna_database = {
    "Actinobacteria": actino_dna,
    "Cat": cat_dna,
    "Dog": dog_dna,
    "E. Coli": ecoli_dna,
    "Fruit Fly": fruit_fly_dna,
    "Horse": horse_dna,
    "Human": human_dna,
    "Mouse": mouse_dna,
    "Plasmodium Falciparum": plasmodium_dna,
    "Thale Cress": thale_cress_dna,
}

mystery_fasta_path = f"{base_path}/data/mystery_dna.fasta"
mystery_dna = get_dna_seq(mystery_fasta_path)

# 4. Compare the GC contents

In [101]:
print(f"================= GC Contents: =================")

print_gc_content(mystery_dna, "Mystery DNA")

for organism, dna_seq in dna_database.items():
    dna_len = len(dna_seq)
    print_gc_content(dna_seq, organism)

================= GC Contents: =================
Mystery DNA: 51.72%
Actinobacteria: 72.50%
Cat: 46.51%
Dog: 34.62%
E. Coli: 51.15%
Fruit Fly: 51.14%
Horse: 45.28%
Human: 37.26%
Mouse: 52.33%
Plasmodium Falciparum: 31.59%
Thale Cress: 34.61%


# 5. k-mer Comparisons
You can first try looking at specific k-mers and their frequency in the mystery DNA vs the other DNA sequences

In [102]:
print(f"================= k-mer Counts: =================")
k = 2       # We set the length k that we want to use
kmer = "AT" # Looking for this k-mer count

mystery_len = len(mystery_dna)
comparison_len = min(100, mystery_len)

mystery_kmers, mystery_total_kmer_count = count_kmers(mystery_dna[:comparison_len], k)
print_kmer_count(get_kmer_percentage(mystery_kmers[kmer], mystery_total_kmer_count),
                 "Mystery DNA",
                 f"k-mer: '{kmer}'")

for organism, dna_seq in dna_database.items():
    dna_len = len(dna_seq)
    comparison_len = min(100, mystery_len, dna_len)
    kmers, total_kmer_count = count_kmers(dna_seq[:comparison_len], k)
    print_kmer_count(get_kmer_percentage(kmers[kmer], total_kmer_count),
                     organism,
                     f"k-mer: '{kmer}'")

================= k-mer Counts: =================
Mystery DNA: 2.04%  (k-mer: 'AT')
Actinobacteria: 3.06%  (k-mer: 'AT')
Cat: 11.22%  (k-mer: 'AT')
Dog: 7.14%  (k-mer: 'AT')
E. Coli: 5.10%  (k-mer: 'AT')
Fruit Fly: 2.04%  (k-mer: 'AT')
Horse: 0.00%  (k-mer: 'AT')
Human: 8.16%  (k-mer: 'AT')
Mouse: 3.06%  (k-mer: 'AT')
Plasmodium Falciparum: 10.20%  (k-mer: 'AT')
Thale Cress: 10.20%  (k-mer: 'AT')


Next, you can try looking for the k-mer that appears most frequently in the mystery DNA and how often it appears in the other DNA sequences.

<b>Bonus:</b>
It might appear that the percentages for the other DNA sequences are significantly smaller than for the myster DNA. What does this tell you about the length of the mystery DNA compared to the other sequences?

In [103]:
k = 3   # We set the length k that we want to use

comparison_len = min(100, mystery_len)

mystery_kmers, mystery_total_kmer_count = count_kmers(mystery_dna[:comparison_len], k)

# Grab kmer that appears most frequently in mystery DNA
max_mystery_kmer, max_mystery_kmer_count = max(mystery_kmers.items(), key=lambda item: item[1])
print_kmer_count(get_kmer_percentage(max_mystery_kmer_count, mystery_total_kmer_count),
                 "Mystery DNA",
                 f"k-mer: '{max_mystery_kmer}'")

for organism, dna_seq in dna_database.items():
    dna_len = len(dna_seq)
    comparison_len = min(100, mystery_len, dna_len)
    kmers, total_kmer_count = count_kmers(dna_seq[:comparison_len], k)
    print_kmer_count(get_kmer_percentage(kmers[max_mystery_kmer], total_kmer_count),
                     organism,
                     f"k-mer: '{max_mystery_kmer}'")

Mystery DNA: 7.22%  (k-mer: 'CTT')
Actinobacteria: 1.03%  (k-mer: 'CTT')
Cat: 0.00%  (k-mer: 'CTT')
Dog: 1.03%  (k-mer: 'CTT')
E. Coli: 2.06%  (k-mer: 'CTT')
Fruit Fly: 2.06%  (k-mer: 'CTT')
Horse: 0.00%  (k-mer: 'CTT')
Human: 1.03%  (k-mer: 'CTT')
Mouse: 1.03%  (k-mer: 'CTT')
Plasmodium Falciparum: 2.06%  (k-mer: 'CTT')
Thale Cress: 2.06%  (k-mer: 'CTT')


# 6. Compare Sequence Alignment Scores
Since the sequences are very long, it would take too long to align them entirely. Instead, we take a substring of each sequence using `comparison_len`.

By default, `comparison_len` is set to `DEFAULT_LEN` ( = 100), unless the mystery or other DNA sequence happens to be smaller than that. The subtring we take starts from the first character in the sequence and ends at index `comparison_len` (in other words, we take the first `comparison_len` # of characters).

Feel free to experiment with:
- `DEFAULT_LEN` to change how long the substring you take is
- Where you take the substring from (hint: instead of taking the first X# of characters, you can take the last X# of characters)

In [104]:
# Scoring Scheme:
MATCH = 1
MISMATCH = -1
GAP_PENALTY = -1

print(f"================= Global Alignment Scores: =================")

DEFAULT_LEN = 100 # Try changing this to get different alignments

for organism, dna_seq in dna_database.items():
    dna_len = len(dna_seq)
    comparison_len = min(DEFAULT_LEN, mystery_len, dna_len)
    S_aligned, T_aligned, max_score = get_alignment(mystery_dna[:comparison_len], dna_seq[:comparison_len])

    print(f"{organism}: {max_score}")
    print(f"{S_aligned}")
    print(f"{T_aligned}\n")

================= Global Alignment Scores: =================
Actinobacteria: 3
GCGCTTGTGTCTCT-GG-AA-GTTGTC--CTTAACGAACTTAGCAAAT-CCCTT--C-GCCTCGCCCG---CCGGCGTGGATACGGTG--TTTCCTTGCTTG-TCCTTTTTGTTC
AAGC--ACGAC-CTGGGCAACCTCG-CGGCCT-GC-ACCGT--C-GATGCGCTTGCCGGCCTCACCCGCACCCGGC-TGAAGAAGATGCAGTACC-GGCCCGAT-CTCCTCG---

Cat: -7
GCGCTT--GTGTCTCTGGAAGTTGTCCTTAACGAACT-TAGCAAATCCCTTCG-CCTCG-CCCGCCGGCGTGGATAC-GG--TGT-TTCCTTGCTTGTC--CT-T-TTTGTTC
G-AATTCACACTATTTGGAA-TGGT-CTGAAAAAAATGT--C-TAT--TTTGGTCC-AGAAAATCC--CAT--TTCCAGGAATCTCTCCCTAGATAGTCAGATATAACTGGGG

Dog: -3
-G---CG-CTTG-T--GT--C--T--CTGGA--AGTTGTCCTTAACGAACTT-AGCAAATCCCTTCGCCTCGCCCGCCGGCGTGGATACGGTGTTTC-CTTGCTTGTCCTTTTTG-TTC
TGATAGGAAATGCTCCGTAACTGTAACTGGACTAGAGGTCATTTAGGGA-TTAAGC--A-CAGGT-GCCT-TCCC-CAAGAAT--CTA--G-GTTACACATAC-TG------TTGCATA

E. Coli: 1
-GC---GC-TTGTG--T-C-TCTGG-AAGT-TGTC-CT-TAACGAACTTAGCAAATCCCT-TC-GCCTCGCCCGCCGGCGTGGATACGGTGTTTCCTTGCTTGTCCTTTTTGTTC
AGCTTTTCATTCTGACTGCAACGGGCAA-TATGTCTCTGT-GTGGA-TTAAAAAAAGAGTGTCTG-ATAG-CAG-

# 7. Conclusion
Now, based on all of the results you gathered, which organism do you think the mystery DNA comes from?

Make sure to thoroughly explain your experiments and reasoning!